In [ ]:
# imports
import pandas as pd
from tqdm import tqdm
import numpy as np
import json
import os
from openai import OpenAI, AsyncOpenAI, OpenAIError
import asyncio
from tqdm.asyncio import tqdm_asyncio

from miti_global_scores import MAIN_PROMPT, MITI_GUIDE

In [ ]:
scripts = pd.read_csv("output/validation_bheavioral_scores_extracted.csv")
api_key = os.getenv("OPENAI_API_KEY")
id_name = "source_pdf"

In [ ]:
scripts.head()

In [ ]:
def make_transcript_df(df):
    # Prefix content based on P-or_C column
    label_map = {'P': 'Client:', 'C': 'Clinician:'}
    # Create prefixed content
    prefixed_content = df.apply(lambda row: f"{label_map.get(row['P_or_C'], '')} {row['Content']}", axis=1)
    # Concatenate for full transcript per source_pdf
    full_transcript = prefixed_content.groupby(df['source_pdf']).apply(lambda x: '\n'.join(x))
    # Output dataframe
    transcript_df = pd.DataFrame({
        'full_transcript': full_transcript.values,
        'source_pdf': full_transcript.index
    })
    return transcript_df

# Example usage
session_mi = make_transcript_df(df = scripts)

In [ ]:
session_mi.head()

In [ ]:
params = {
    "model": "gpt-5-nano-2025-08-07",
    "text": {"verbosity": "medium"}, # low, medium, high
    "reasoning": {"effort": "low"}, # minimal, low, medium, high
    "store": False,
    "max_output_tokens": 5000,
}

In [ ]:
################################################
#####       API Queries         ################
################################################

# Setup OpenAI
client = OpenAI(api_key=api_key)

# Prepare the input data for parallel coding (N x M requests, N = number of sessions, M = number of MITI codes)
input_data = []
for _, row in tqdm(session_mi.iterrows(), total=session_mi.shape[0]):
    for component, instructions in MITI_GUIDE.items():
        identifier = row[id_name]
        prompt = MAIN_PROMPT.format(
            transcript=row["full_transcript"],
            component_name=component,
            coding_instructions=instructions)
        input_data.append([identifier, component, prompt])

input_data = pd.DataFrame(input_data)
input_data.columns = [id_name, "miti_dimension", "prompt"]


# PARALLEL REQUESTS
client = AsyncOpenAI(api_key=api_key)

async def classify_row(row, sem, params):
    async with sem:
        identifier = row[id_name]
        component = row["miti_dimension"]
        prompt = row["prompt"]
        try:
            reply = await client.responses.create(input=prompt, **params)
            response = reply.model_dump()
            response_dict = json.loads(response["output"][1]["content"][0]["text"])
            return [identifier, component, response, response_dict["score"], response_dict["justification"], prompt, "0"]
        except Exception as e:
            return [identifier, component, "", -999, -999, prompt, str(e)]

async def main(data, params, n_workers):
    """Execute parallel API requests"""
    sem = asyncio.Semaphore(n_workers)
    tasks = [classify_row(row, sem, params) for _, row in data.iterrows()]
    results = []
    errors = []

    for step in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
        result = await step
        if isinstance(result, list):
            results.append(result)
        else:
            errors.append(result)

    return results, errors


# run (the code below is for running in a jupyter notebook, hence the "await")
results, errors = await main(input_data, params, n_workers=8)

# Store results
df = pd.DataFrame(results)
df.columns = [id_name, "miti_dimension", "gpt_response", "score", "justification", "prompt", "error"]



In [ ]:
date = pd.Timestamp.now().strftime("%Y%m%d")
df.to_csv(f"output/validation_miti_global_scores_{date}.csv", index=False)

In [ ]:
df.tail()